**STEP 1: Simulation**

In [1]:
import numpy as np
import random
from scipy.stats import wishart
def simulate_matrix(n, d, no_components,rho,abs_bound_X):
    ###Form the X matrix
    X = np.zeros((0, d))
    # Use base_value and reminder to evenly distributed the number of samples in each component
    def generate_components(n, no_components):
        base_value = n // no_components
        remainder = n % no_components
        components = [base_value] * no_components
        for i in range(remainder):
            components[i] += 1
        random.shuffle(components)
        if sum(components) > n:
            components[n-1] -= sum(components) - n
        elif sum(components) < n:
            components[n-1] += n - sum(components)
        return components
    # Generate these multivariate gaussian components in different levels of mean and cov.
    num_list = generate_components(n, no_components)
    for i in range(no_components):
        # each level's mean value, grow by 4's power.
        mean_value = 5**(i+1)
        # 0.8 probability of 1 and 0.2 probability of 0 for creating a sparse mean vector.
        mean_vector = np.random.binomial(1, 0.8, size = d)
        mean = mean_value * mean_vector
        # randomly generate a cov matrix and make sure that it is symmetric.
        cov = wishart.rvs(df=d, scale=np.eye(d))
        # add a small value to the diagonal to ensure positive-definiteness
        cov += np.eye(d) * 1e-6
        # generate a component of X.
        X_ = np.random.multivariate_normal(mean, cov, num_list[i])
        X_ = np.array(X_)
        # append the component to the list.
        X = np.vstack((X, X_))
    X = np.array(X)
    # Set the elements < 3 to 0 to make it a sparse matrix
    X[abs(X) < abs_bound_X] = 0
    ###Form the fixed beta
    np.random.seed(39)
    beta_value = np.random.binomial(1, 0.8, size = d)
    beta = (beta_value * np.random.uniform(0, 1, size = d)).reshape(-1,1)
    beta = np.array(beta)
    np.random.seed(None)
    ###Form the error term - epsilon, which has the auto-correlation structure in time series analysis.
    r = rho # autocorrelation coefficient
    epsilon = np.zeros((n,1))
    epsilon[0] = np.random.normal(5,1)
    for t in range(1,n):
        epsilon[t] = r * epsilon[t-1] + np.random.normal(5,1)
    epsilon = np.array(epsilon)
    ###Form the Y matrix
    Y = np.array(X @ beta + epsilon)
    ###Count the number of elements < 0.1 in X and Y
    return X,Y,beta,epsilon

# Matrix Settled
n = 3000       # number of rows         
d = 200        # number of columns
no_components = 4 # number of levels of multivariate gaussian distribution in X
rho = 0.8 # autocorrelation coefficient in time series analysis
abs_bound_X = 2 # absolute bound for X to set elements to be 0
X,Y,beta,epsilon = simulate_matrix(n, d, no_components,rho,abs_bound_X)
print("The simulated matrix X is:")
print(X)
print("The simulated vector Y is:")
print(Y)

The simulated matrix X is:
[[ 27.26310623  -3.44156151  15.72856437 ... -22.35054714   2.8346666
    3.50478905]
 [ 34.6262137  -14.68521227  -2.33135438 ...  14.50572207  -2.35624877
   -3.36388004]
 [  0.          21.55587283  23.88761634 ...  -9.8461337  -22.09510323
    5.64082011]
 ...
 [ -8.788455     0.         636.97381851 ... 637.84402158 640.42824725
  617.8858743 ]
 [ 25.8105679    5.33602706 645.28830288 ... 625.24231783 583.64643783
  630.9815448 ]
 [ -4.54382937  23.37398929 607.84759813 ... 617.48450967 595.67840637
  607.18668977]]
The simulated vector Y is:
[[  364.27544335]
 [  367.54606972]
 [  259.29560652]
 ...
 [35873.38345523]
 [35847.92614186]
 [35786.77909879]]


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import lsqr
from scipy.linalg import cho_factor, cho_solve
def solver(A,b,solver):
    if solver == 'lsqr':
    # lsqr algorithm
        X = csr_matrix(A)
        Y = np.array(b)
        x_star = lsqr(X, Y)[0]
    # cholesky decomposition algorithm
    elif solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b

    return x_star
x_star_lsqr = solver(X,Y,'lsqr')
x_star_cholesky = solver(X,Y,'cholesky')
x_star_direct = solver(X,Y,'direct')
norm_lsqr = calculate_the_norm_square(X,Y,x_star_lsqr)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:")
print(norm_lsqr, norm_cholesky, norm_direct)
minimum_norm_square = min(norm_lsqr, norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)

The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:
3744313369816596.0 149791.95348244606 149791.95348244492
Here we output the minimum of the norm suqare for x_star calculated as:
149791.95348244492


In [4]:
###From the above different norm and optimal solver calculations, it is obvious that:
###the direct solver and cholesky decomposition are the optimal solvers.
###Since their norms are very very very nearly the same
###we choose the direct solver as our optimal solver when matrix is not so large.
###And we choose the cholesky decomposition as our optimal solver when matrix is so large.
x_Star = x_star_direct
norm_Star = norm_direct
print("The optimal solver is:")
print(x_Star)

The optimal solver is:
[[ 6.88691075e-01]
 [ 7.00059803e-01]
 [ 2.44241964e-02]
 [ 3.92724533e-01]
 [ 2.12908835e-01]
 [ 6.98300434e-01]
 [ 5.25790430e-01]
 [ 7.55935700e-02]
 [ 5.28277307e-01]
 [-8.39366804e-03]
 [-7.93555968e-02]
 [-1.62786707e-02]
 [ 7.40544577e-03]
 [ 1.01344274e+00]
 [ 1.61822321e-02]
 [ 8.02165195e-03]
 [ 4.76083930e-01]
 [ 1.25469453e-01]
 [ 4.37366990e-01]
 [-1.35374123e-02]
 [ 5.37481501e-01]
 [ 2.64930011e-02]
 [ 8.84691749e-02]
 [ 8.65377107e-01]
 [ 9.52286365e-01]
 [-1.08980039e-02]
 [ 7.00558317e-01]
 [ 4.90365359e-01]
 [ 1.40138785e-01]
 [ 3.73969764e-02]
 [ 2.38621334e-01]
 [ 7.92313787e-01]
 [ 2.09129369e-01]
 [ 4.21595138e-02]
 [ 7.96682150e-02]
 [ 5.55636592e-01]
 [ 3.18853120e-02]
 [ 3.78615991e-02]
 [-4.40791357e-02]
 [ 6.34949933e-03]
 [ 2.57765831e-01]
 [ 6.12303542e-01]
 [ 6.30676851e-01]
 [ 6.44054888e-03]
 [ 8.88988177e-01]
 [ 5.31970105e-01]
 [ 4.50512861e-01]
 [ 6.22023483e-01]
 [ 6.54228123e-01]
 [ 5.35143202e-01]
 [ 1.45596613e-01]
 [ 9.443

**STEP 2: Uniform Sampling sketch // Algorithm 1: Distributed Randomized Regression**

In [5]:
# e.g. our desired sketching size is m = 1000
from operator import index


m = 1000

# Algorithm 1 inserting inside uniform sampling: Distributed Randomized Regression

# S_k @ A here is just computed as A [uniformly_sampled_index]
# As S_k here is just a diagnoal matrix of 1 or 0 where sampled rows have 1 as value
def uniform_sampling(A,b,n,m,q):
    x_hat_list = []
    for i in range(q):
        index = np.random.choice(n, size=m, replace=False)
        A_sk = A[index]
        b_sk = b[index]
        x_hat = np.linalg.inv(A_sk.T @ A_sk) @ A_sk.T @ b_sk
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar
